# 🚀 Módulo 1: El Corazón del Rendimiento — LRU Cache
## Notebook de Conocimiento Guiado

**Objetivo:** Dominar la LRU Cache implementándola desde cero, entendiendo
cada decisión de diseño y aplicándola en escenarios de entrevista reales.

### 🗺️ Ruta de aprendizaje
1. ¿Qué problema resuelve una cache?
2. Estructura interna: HashMap + Lista Doblemente Enlazada
3. Implementación paso a paso
4. Ejercicios guiados
5. Desafíos de entrevista

> 🎮 **Lore:** Eres el ingeniero principal de una nave espacial controlada por IA.
> La nave necesita un sistema de memoria para coordenadas de navegación que opere en **O(1)**.


## 📚 Parte 1: ¿Por qué necesitamos una LRU Cache?

Una **cache** almacena resultados costosos de calcular para devolverlos rápidamente.
La política **LRU** (Least Recently Used) descarta el elemento **menos recientemente usado**
cuando la cache está llena.

### El problema de las estructuras simples

| Estructura | get() | put() | Orden de acceso |
|-----------|-------|-------|-----------------|
| Lista      | O(n)  | O(1)  | ✅ Sí           |
| Diccionario| O(1)  | O(1)  | ❌ No           |
| **LRU Cache** | **O(1)** | **O(1)** | **✅ Sí** |

Necesitamos lo mejor de ambos mundos → **HashMap + Lista Doblemente Enlazada**.


In [ ]:
# 🔍 Visualización del problema
# ¿Qué ocurre cuando solo usamos un diccionario?

cache_simple = {}
orden_acceso = []  # Tenemos que mantener esto manualmente

def get_simple(key):
    if key in cache_simple:
        # Mover al frente del orden... pero esto es O(n) con una lista normal
        orden_acceso.remove(key)
        orden_acceso.append(key)
        return cache_simple[key]
    return -1

def put_simple(key, value, capacidad=3):
    if key not in cache_simple and len(cache_simple) >= capacidad:
        # Encontrar el LRU: O(1) solo si tenemos la estructura correcta
        lru = orden_acceso.pop(0)  # O(n) con lista normal!
        del cache_simple[lru]
        print(f"  ❌ Eliminando LRU: key={lru}")
    cache_simple[key] = value
    if key in orden_acceso:
        orden_acceso.remove(key)
    orden_acceso.append(key)

# Demostración del problema
put_simple(1, "Alpha")
put_simple(2, "Beta")
put_simple(3, "Gamma")
print(f"Cache: {cache_simple}")
print(f"Orden (izq=LRU, der=MRU): {orden_acceso}")

get_simple(1)  # Accedemos a Alpha → se mueve al frente
print(f"Tras acceder a 1: {orden_acceso}")

put_simple(4, "Delta")  # Delta entra → Beta (LRU) sale
print(f"Tras insertar 4: {dict(cache_simple)}")
print(f"Orden final: {orden_acceso}")
print()
print("⚠️  El problema: remove() en lista es O(n). Necesitamos algo mejor.")


## 🏗️ Parte 2: La Solución — Lista Doblemente Enlazada

### ¿Por qué doblemente enlazada?

```
         ┌──────────┐     ┌──────────┐     ┌──────────┐
HEAD ←→  │ key=1    │ ←→  │ key=3    │ ←→  │ key=2    │  ←→ TAIL
         │ val=100  │     │ val=300  │     │ val=200  │
         └──────────┘     └──────────┘     └──────────┘
           [MRU]                              [LRU]
```

- **Enlace hacia atrás (prev):** Permite eliminar en O(1) sin recorrer la lista
- **Nodos centinela HEAD y TAIL:** Eliminan los casos borde de lista vacía

### La combinación mágica:
- **HashMap:** `key → Node` — acceso directo en O(1)
- **Lista DL:** orden de uso — reordenamiento en O(1)


In [ ]:
# 🔨 IMPLEMENTACIÓN PASO A PASO
# Paso 1: El Nodo de la Lista

class Node:
    """Nodo de la lista doblemente enlazada."""
    __slots__ = ("key", "value", "prev", "next")
    
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None
    
    def __repr__(self):
        return f"Node(k={self.key}, v={self.value})"

# Visualizar la lista
def visualizar_lista(head, tail):
    elementos = []
    current = head.next
    while current != tail:
        elementos.append(f"[{current.key}:{current.value}]")
        current = current.next
    return "HEAD ←→ " + " ←→ ".join(elementos) + " ←→ TAIL" if elementos else "HEAD ←→ TAIL (vacía)"

# Crear cabecera y cola centinelas
head = Node()  # Sentinel HEAD
tail = Node()  # Sentinel TAIL
head.next = tail
tail.prev = head

print("Lista inicial:")
print(visualizar_lista(head, tail))
print()

# Insertar nodo al frente (MRU)
def add_to_front(head, node):
    first = head.next
    node.prev = head
    node.next = first
    head.next = node
    first.prev = node

n1 = Node(key=1, value=100)
n2 = Node(key=2, value=200)
n3 = Node(key=3, value=300)

add_to_front(head, n1)
add_to_front(head, n2)
add_to_front(head, n3)

print("Tras insertar 1, 2, 3 al frente:")
print(visualizar_lista(head, tail))
print("(El último insertado es el MRU → izquierda)")


In [ ]:
# Paso 2: Operación _remove — O(1)
def remove(node):
    """Desenlaza un nodo de la lista. O(1)."""
    prev_node = node.prev
    next_node = node.next
    prev_node.next = next_node
    next_node.prev = prev_node

print("Antes de eliminar n2 (key=2):")
print(visualizar_lista(head, tail))

remove(n2)
print("Después:")
print(visualizar_lista(head, tail))
print()

# Ahora podemos mover un nodo al frente en O(1):
# 1. remove(node) → desenlazar
# 2. add_to_front(head, node) → reinsertar al frente
print("Accedemos a n1 (key=1) → moverlo al frente (MRU):")
remove(n1)
add_to_front(head, n1)
print(visualizar_lista(head, tail))


In [ ]:
# 🎯 IMPLEMENTACIÓN COMPLETA DE LRU CACHE

class LRUCache:
    def __init__(self, capacity: int):
        if capacity < 1:
            raise ValueError("La capacidad debe ser >= 1")
        self.capacity = capacity
        self.cache = {}  # key → Node
        
        # Centinelas (nunca contienen datos reales)
        self.head = Node()
        self.tail = Node()
        self.head.next = self.tail
        self.tail.prev = self.head
    
    def _remove(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev
    
    def _add_to_front(self, node):
        first = self.head.next
        node.prev = self.head
        node.next = first
        self.head.next = node
        first.prev = node
    
    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        node = self.cache[key]
        # Mover al frente: ahora es el MRU
        self._remove(node)
        self._add_to_front(node)
        return node.value
    
    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            node = self.cache[key]
            node.value = value
            self._remove(node)
            self._add_to_front(node)
        else:
            if len(self.cache) >= self.capacity:
                # Evictar LRU (nodo justo antes de TAIL)
                lru = self.tail.prev
                self._remove(lru)
                del self.cache[lru.key]
                print(f"  🗑️  Evictado: key={lru.key}")
            new_node = Node(key, value)
            self.cache[key] = new_node
            self._add_to_front(new_node)
    
    def __repr__(self):
        items = []
        current = self.head.next
        while current != self.tail:
            items.append(f"{current.key}:{current.value}")
            current = current.next
        return f"LRU[{' → '.join(items)}] (MRU→LRU)"

# DEMO
print("=== Demo de LRU Cache (capacidad=3) ===")
lru = LRUCache(3)
lru.put(1, 100); print(f"put(1,100): {lru}")
lru.put(2, 200); print(f"put(2,200): {lru}")
lru.put(3, 300); print(f"put(3,300): {lru}")
lru.get(1);       print(f"get(1)=100: {lru}  ← key=1 ahora es MRU")
lru.put(4, 400);  print(f"put(4,400): {lru}")  # key=2 es LRU → evictar
print(f"get(2)={lru.get(2)}  ← key=2 fue evictada")


## 🏋️ Ejercicios Guiados

### Ejercicio 1: Implementar `peek()` — consultar sin actualizar el orden

Muchas veces queremos ver un valor sin cambiar el orden de la cache
(por ejemplo, para depuración o monitoreo).


In [ ]:
# ✏️ EJERCICIO 1: Implementar peek(key) sin modificar el orden LRU
# peek() debe devolver el valor si existe, o -1 si no existe,
# pero SIN mover el nodo al frente de la lista.

class LRUCacheConPeek(LRUCache):
    def peek(self, key: int) -> int:
        # TODO: implementa esta función
        # Pista: es como get() pero sin las llamadas _remove()/_add_to_front()
        pass

# Tests para verificar tu implementación:
lru2 = LRUCacheConPeek(3)
lru2.put(1, 100)
lru2.put(2, 200)
lru2.put(3, 300)

print(f"Estado inicial: {lru2}")
valor = lru2.peek(1)  # key=1 está en cache pero NO debe moverse al frente
print(f"peek(1) = {valor}  (debe ser 100)")
print(f"Estado tras peek: {lru2}")
print("(key=1 debe seguir siendo LRU, NO haberse movido al frente)")

# Forzar evicción — si peek no movió el orden, key=1 debe evictarse
lru2.put(4, 400)
print(f"Tras put(4,400): {lru2}")
print(f"get(1) = {lru2.get(1)}  (debe ser -1 si peek funcionó correctamente)")


In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1
class LRUCacheConPeek(LRUCache):
    def peek(self, key: int) -> int:
        if key not in self.cache:
            return -1
        # Solo leer el valor, SIN mover el nodo
        return self.cache[key].value

# Verificación
lru2 = LRUCacheConPeek(3)
lru2.put(1, 100); lru2.put(2, 200); lru2.put(3, 300)
print(f"Estado: {lru2}")
print(f"peek(1) = {lru2.peek(1)}  ✅")
print(f"Estado tras peek: {lru2}  (key=1 sigue siendo LRU)")
lru2.put(4, 400)
print(f"Tras put(4): {lru2}  (key=1 evictado correctamente)")
assert lru2.get(1) == -1, "peek() no debió mover el orden"
print("✅ Test pasado!")


### Ejercicio 2: LRU Cache con TTL (Time-To-Live)

Implementa una versión donde cada entrada expira después de N segundos.
Es el tipo de pregunta que aparece en entrevistas de Google y Cloudflare.


In [ ]:
import time as _time

# ✏️ EJERCICIO 2: LRU Cache con TTL
class LRUCacheTTL(LRUCache):
    """LRU Cache donde cada entrada expira tras ttl_seconds segundos."""
    
    def __init__(self, capacity: int, ttl_seconds: float):
        super().__init__(capacity)
        self.ttl = ttl_seconds
        self.timestamps = {}  # key → timestamp de inserción
    
    def put(self, key: int, value: int) -> None:
        # TODO: llama a super().put() y guarda el timestamp actual
        pass
    
    def get(self, key: int) -> int:
        # TODO: verificar si la entrada expiró antes de devolverla
        # Si expiró → eliminarla y devolver -1
        # Pista: usa _time.time() para el tiempo actual
        pass

# Tests básicos (se ejecutan con timestamps reales)
cache_ttl = LRUCacheTTL(capacity=3, ttl_seconds=0.5)
cache_ttl.put(1, 100)
print(f"get(1) inmediato = {cache_ttl.get(1)}  (debe ser 100)")
print("Esperando 0.6 segundos...")
_time.sleep(0.6)
print(f"get(1) tras expirar = {cache_ttl.get(1)}  (debe ser -1)")


In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2
class LRUCacheTTL(LRUCache):
    def __init__(self, capacity: int, ttl_seconds: float):
        super().__init__(capacity)
        self.ttl = ttl_seconds
        self.timestamps = {}
    
    def put(self, key: int, value: int) -> None:
        super().put(key, value)
        self.timestamps[key] = _time.time()
    
    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        if _time.time() - self.timestamps.get(key, 0) > self.ttl:
            # Expirado: eliminar de la cache
            node = self.cache[key]
            self._remove(node)
            del self.cache[key]
            del self.timestamps[key]
            return -1
        return super().get(key)

# Verificación
cache_ttl = LRUCacheTTL(3, 0.3)
cache_ttl.put(1, 100)
assert cache_ttl.get(1) == 100, "Debe estar en cache"
_time.sleep(0.4)
assert cache_ttl.get(1) == -1, "Debe haber expirado"
print("✅ LRU Cache con TTL funciona correctamente!")


## 🎯 Quiz Final

Preguntas de entrevista reales sobre LRU Cache:

1. **¿Por qué O(1) y no O(log n)?** ¿Qué estructura daría O(log n)?
2. **¿Qué pasa si `capacity=0`?** ¿Tu implementación lo maneja?
3. **¿Cómo harías una LRU Cache thread-safe?** (pista: `threading.Lock`)
4. **¿Cuándo elegirías LFU (Least Frequently Used) sobre LRU?**
5. **¿Cómo implementarías una LRU Cache distribuida?** (pista: Redis)

### 📊 Complejidad — Resumen

| Operación | Tiempo | Espacio |
|-----------|--------|---------|
| `get(key)` | **O(1)** | O(1) |
| `put(key, val)` | **O(1)** | O(1) |
| `peek(key)` | **O(1)** | O(1) |
| Espacio total | — | **O(n)** |

### 🏆 ¡Felicidades!
Has dominado la LRU Cache. Este patrón aparece en Redis, Memcached,
los cachés de CPU, los cachés de páginas de sistemas operativos, y la
implementación de `functools.lru_cache` en Python.
